# 02_feature_engineering.ipynb

Extrae 72 features numericas por muestra de cada fuente auditada en 01_data_audit.
Cada fuente se guarda como parquet independiente en `data/processed/`.
El schema es uniforme: `sample_id`, 72 features nombradas, `label` (int), `timestamp`.

Labels:
- 0: legitimate
- 1: sqli
- 2: xss
- 3: path_traversal
- 4: command_injection

Fuentes procesadas en este notebook:
1. Payloads.csv
2. payload_full.csv
3. command_injection.csv
4. XSS_dataset.csv
5. data_capec_multilabel.csv
6. modsec-learn JSON
7. PT wordlists (Dp.txt)
8. OWASP logs
9. RussellMitchell (legitimate only)

In [15]:
import pandas as pd
import numpy as np
import re, json, html as html_lib
from pathlib import Path
from urllib.parse import unquote, urlparse
from collections import Counter

BASE_DIR      = Path("../data")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

TODAY = pd.Timestamp("2026-06-08")

LABEL = {
    "legitimate": 0, "sqli": 1, "xss": 2,
    "path_traversal": 3, "command_injection": 4,
}

# Columnas canonicas en orden (72 features)
FEATURE_COLS = [
    # Grupo 1: longitudes
    "payload_length", "payload_entropy", "uri_length", "path_length",
    "query_string_length", "body_length", "body_entropy",
    "path_depth", "query_param_count", "fragment_present",
    # Grupo 2: composicion de caracteres
    "special_char_ratio", "numeric_char_ratio", "uppercase_ratio",
    "whitespace_count", "newline_char_count", "null_byte_count",
    "extended_ascii_ratio", "payload_token_count",
    # Grupo 3: encoding
    "url_encoded_ratio", "encoded_char_freq", "double_encoded_count",
    "hex_escape_count", "unicode_escape_count", "html_entity_count", "base64_like_count",
    # Grupo 4: SQLi
    "sqli_keyword_count", "sqli_keyword_density", "sqli_comment_count",
    "sqli_operator_count", "quote_count", "semicolon_count", "parenthesis_count",
    "union_present", "select_present",
    # Grupo 5: XSS
    "xss_marker_count", "xss_marker_density", "html_tag_count",
    "script_tag_present", "js_event_handler_count", "javascript_url_count",
    "html_entity_density", "alert_function_present", "inline_style_present",
    # Grupo 6: Path Traversal
    "traversal_sequence_count", "path_separator_count", "absolute_path_indicator",
    "sensitive_file_target", "sensitive_extension_count", "file_extension_suspicious",
    "dotdot_encoded_count",
    # Grupo 7: Command Injection
    "pipe_count", "backtick_count", "shell_command_count",
    "command_separator_count", "redirect_operator_count",
    "dollar_sign_count", "subshell_count", "os_path_indicator",
    # Grupo 8: HTTP request
    "method_is_get", "method_is_post", "ua_present", "ua_length",
    "ua_suspicious", "content_type_encoded", "authorization_length",
    "unusual_headers_count", "status_code",
    # Grupo 9: temporal (0 para fuentes sin timestamp de sesion)
    "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s",
]

assert len(FEATURE_COLS) == 72, f"Esperado 72, hay {len(FEATURE_COLS)}"
print(f"Setup OK | PROCESSED_DIR={PROCESSED_DIR} | features={len(FEATURE_COLS)}")

Setup OK | PROCESSED_DIR=..\data\processed | features=72


# Seccion 2: Funciones de extraccion de features

Todas las funciones son vectorizadas (operan sobre pd.Series completas).
`extract_features()` acepta la serie de payloads como argumento obligatorio
y series opcionales para campos HTTP. Series ausentes se rellenan con 0/vacio.

In [16]:
# --- Patrones compilados ---
# Nota: patrones usados en str.contains() usan grupos no-capturantes (?:...)
# para evitar UserWarning de pandas. Patrones en str.count() pueden usar (...).

_SQL_KW = re.compile(
    r"(?i)\b(select|union|insert|update|delete|drop|create|alter|exec(?:ute)?"
    r"|where|from\b|into\b|null\b|like\b|between|exists|having|order\s+by"
    r"|group\s+by|cast\b|convert\b|char\b|varchar|concat|count\b|sleep\b"
    r"|benchmark|substr(?:ing)?|mid\b|ascii\b|hex\b|unhex|load_file"
    r"|outfile|dumpfile|information_schema|schema\b|database\(|version\(|user\()"
)

_XSS_PAT = re.compile(
    r"(?i)(<\s*script|<\s*/\s*script|<\s*img|<\s*svg|<\s*iframe|<\s*body"
    r"|<\s*input|on(?:error|load|click|mouseover|mouseout|focus|input|keyup|keydown)\s*="
    r"|javascript\s*:|alert\s*\(|confirm\s*\(|prompt\s*\(|document\.cookie"
    r"|window\.location|eval\s*\(|innerHTML|src\s*=\s*[\"']?\s*javascript)"
)

_TRAV = re.compile(
    r"(?i)(\.\.[\\/]|%2e%2e[%\\/]|%252e%252e|%c0%ae%c0%ae|\.\.%2f|\.\.%5c"
    r"|\.\./|\.\.\\)"
)
_DOTDOT_ENC = re.compile(r"(?i)(%2e%2e|%252e%252e|%c0%ae)")

# str.contains() -> usar (?:...) para evitar UserWarning
_SENS_FILE = re.compile(
    r"(?i)(?:etc/passwd|etc/shadow|etc/hosts|win\.ini|boot\.ini"
    r"|\.htaccess|\.htpasswd|wp-config\.php|\.git/config|\.env\b"
    r"|\.bash_history|/proc/self|web\.config|php\.ini)"
)
_SENS_EXT  = re.compile(r"(?i)\.(conf|ini|log|bak|env|backup|old|sql|db)\b")
_SUSP_EXT  = re.compile(r"(?i)\.(php\d?|aspx?|jspx?)\b")

_CMD_KW = re.compile(
    r"(?i)\b(cat\b|ls\b|dir\b|id\b|whoami|wget\b|curl\b|bash\b|sh\b"
    r"|chmod|chown|rm\b|cp\b|mv\b|ping\b|nc\b|ncat\b|netcat|python|perl"
    r"|ruby|php\b|powershell|cmd\.exe|\/bin\/|\/etc\/passwd|\/etc\/shadow)"
)
_CMD_SEP  = re.compile(r"(&&|\|\||[|;`])")
_REDIRECT = re.compile(r"(>>|<<|[><])")
_SUBSHELL = re.compile(r"(\$\(|`[^`]+`)")

_URL_ENC  = re.compile(r"%[0-9a-fA-F]{2}")
_DBL_ENC  = re.compile(r"(?i)%25[0-9a-fA-F]{2}")
_HEX_ESC  = re.compile(r"(?i)(\\x[0-9a-fA-F]{2}|0x[0-9a-fA-F]+)")
_UNI_ESC  = re.compile(r"(?i)(\\u[0-9a-fA-F]{4}|\\U[0-9a-fA-F]{8}|%u[0-9a-fA-F]{4})")
_HTML_ENT = re.compile(r"(?i)(&[a-zA-Z]+;|&#\d+;|&#x[0-9a-fA-F]+;)")
_B64_LIKE = re.compile(r"[A-Za-z0-9+/]{20,}={0,2}")

# str.contains() -> usar (?:...)
_SCANNER_UA = re.compile(
    r"(?i)(?:sqlmap|nikto|dirb|dirbuster|nmap|masscan|nuclei|burpsuite"
    r"|zaproxy|w3af|acunetix|nessus|openvas|metasploit|python-requests"
    r"|go-http|curl/|wget/|libwww-perl|httpclient)"
)

print("Patrones compilados OK")

Patrones compilados OK


In [17]:
def _entropy_series(s: pd.Series) -> pd.Series:
    """Shannon entropy (bits) por fila, vectorizado sobre bytes UTF-8."""
    def _ent(x):
        if not x:
            return 0.0
        b = x.encode("utf-8", errors="replace")
        _, c = np.unique(np.frombuffer(b, dtype=np.uint8), return_counts=True)
        p = c / c.sum()
        return float(-np.sum(p * np.log2(p + 1e-12)))
    return s.fillna("").apply(_ent)


def _ext_ascii_ratio(s: pd.Series) -> pd.Series:
    return s.fillna("").apply(lambda x: sum(1 for c in x if ord(c) > 127) / max(len(x), 1))


def extract_features(
    payload:   pd.Series,
    uri:       pd.Series = None,
    qs:        pd.Series = None,
    method:    pd.Series = None,
    body:      pd.Series = None,
    ua:        pd.Series = None,
    ctype:     pd.Series = None,
    auth_len:  pd.Series = None,
    unusual_h: pd.Series = None,
    status:    pd.Series = None,
) -> pd.DataFrame:
    """
    Extrae 72 features desde payload strings y campos HTTP opcionales.
    Todas las Series deben estar alineadas por indice.
    Series ausentes se inicializan con string vacio o cero segun tipo.
    """
    n = len(payload)
    def _s(x, d=""): return x.fillna(d).astype(str) if x is not None else pd.Series([d]*n, dtype=str, index=payload.index)
    def _n(x, d=0):  return x.fillna(d).astype(float) if x is not None else pd.Series([float(d)]*n, index=payload.index)

    p  = _s(payload)
    u  = _s(uri)
    q  = _s(qs)
    b  = _s(body)
    pl = p.str.len().clip(lower=1).astype(float)   # denominador seguro

    feat = pd.DataFrame(index=payload.index)

    # --- Grupo 1: longitudes ---
    feat["payload_length"]       = p.str.len()
    feat["payload_entropy"]      = _entropy_series(p)
    feat["uri_length"]           = u.str.len()
    feat["path_length"]          = u.str.replace(r"\?.*$", "", regex=True).str.len()
    feat["query_string_length"]  = q.str.len()
    feat["body_length"]          = b.str.len()
    feat["body_entropy"]         = _entropy_series(b)
    feat["path_depth"]           = u.str.replace(r"\?.*$", "", regex=True).str.count("/")
    feat["query_param_count"]    = q.str.count("&") + (q.str.len() > 0).astype(int)
    feat["fragment_present"]     = u.str.contains("#", regex=False).astype(int)

    # --- Grupo 2: composicion de caracteres ---
    feat["special_char_ratio"]   = p.str.count(r"[!@#$%^&*()\[\]{};:'\",./<>?|\\=+_~`]") / pl
    feat["numeric_char_ratio"]   = p.str.count(r"\d") / pl
    feat["uppercase_ratio"]      = p.str.count(r"[A-Z]") / pl
    feat["whitespace_count"]     = p.str.count(r"\s")
    feat["newline_char_count"]   = p.str.count(r"[\n\r]") + p.str.count(r"(?i)%0[da]")
    feat["null_byte_count"]      = p.str.count(r"\x00") + p.str.count(r"(?i)%00")
    feat["extended_ascii_ratio"] = _ext_ascii_ratio(p)
    feat["payload_token_count"]  = p.str.split().str.len().fillna(0)

    # --- Grupo 3: encoding ---
    enc_cnt = p.str.count(_URL_ENC.pattern)
    feat["url_encoded_ratio"]    = enc_cnt / pl
    feat["encoded_char_freq"]    = enc_cnt
    feat["double_encoded_count"] = p.str.count(_DBL_ENC.pattern)
    feat["hex_escape_count"]     = p.str.count(_HEX_ESC.pattern)
    feat["unicode_escape_count"] = p.str.count(_UNI_ESC.pattern)
    feat["html_entity_count"]    = p.str.count(_HTML_ENT.pattern)
    feat["base64_like_count"]    = p.str.count(_B64_LIKE.pattern)

    # --- Grupo 4: SQLi ---
    sqli_kw = p.str.count(_SQL_KW.pattern)
    tok_cnt = feat["payload_token_count"].clip(lower=1)
    feat["sqli_keyword_count"]   = sqli_kw
    feat["sqli_keyword_density"] = sqli_kw / tok_cnt
    feat["sqli_comment_count"]   = p.str.count(r"(--|#(?!\d)|\/\*|\*/)")
    feat["sqli_operator_count"]  = p.str.count(r"(<>|!=|>=|<=|(?<![<>!])=(?!=))")
    feat["quote_count"]          = p.str.count(r"[\'\"]")
    feat["semicolon_count"]      = p.str.count(";")
    feat["parenthesis_count"]    = p.str.count(r"[()]")
    feat["union_present"]        = p.str.contains(r"(?i)\bunion\b", regex=True).astype(int)
    feat["select_present"]       = p.str.contains(r"(?i)\bselect\b", regex=True).astype(int)

    # --- Grupo 5: XSS ---
    xss_cnt = p.str.count(_XSS_PAT.pattern)
    feat["xss_marker_count"]         = xss_cnt
    feat["xss_marker_density"]        = xss_cnt / pl * 100
    feat["html_tag_count"]            = p.str.count(r"<[a-zA-Z/]")
    feat["script_tag_present"]        = p.str.contains(r"(?i)<\s*script", regex=True).astype(int)
    feat["js_event_handler_count"]    = p.str.count(r"(?i)\bon[a-z]{2,20}\s*=")
    feat["javascript_url_count"]      = p.str.count(r"(?i)javascript\s*:")
    feat["html_entity_density"]       = feat["html_entity_count"] / pl * 100
    feat["alert_function_present"]    = p.str.contains(r"(?i)\b(?:alert|confirm|prompt)\s*\(", regex=True).astype(int)
    feat["inline_style_present"]      = p.str.contains(r"(?i)\bstyle\s*=", regex=True).astype(int)

    # --- Grupo 6: Path Traversal ---
    trav_cnt = p.str.count(_TRAV.pattern)
    feat["traversal_sequence_count"]  = trav_cnt
    feat["path_separator_count"]      = p.str.count(r"[/\\]")
    feat["absolute_path_indicator"]   = p.str.contains(r"^[/\\]|^[a-zA-Z]:[/\\]", regex=True).astype(int)
    feat["sensitive_file_target"]     = p.str.contains(_SENS_FILE.pattern, regex=True).astype(int)
    feat["sensitive_extension_count"] = p.str.count(_SENS_EXT.pattern)
    feat["file_extension_suspicious"] = p.str.count(_SUSP_EXT.pattern)
    feat["dotdot_encoded_count"]      = p.str.count(_DOTDOT_ENC.pattern)

    # --- Grupo 7: Command Injection ---
    feat["pipe_count"]               = p.str.count(r"\|")
    feat["backtick_count"]           = p.str.count(r"`")
    feat["shell_command_count"]      = p.str.count(_CMD_KW.pattern)
    feat["command_separator_count"]  = p.str.count(_CMD_SEP.pattern)
    feat["redirect_operator_count"]  = p.str.count(_REDIRECT.pattern)
    feat["dollar_sign_count"]        = p.str.count(r"\$")
    feat["subshell_count"]           = p.str.count(_SUBSHELL.pattern)
    feat["os_path_indicator"]        = p.str.contains(r"(?i)(?:/bin/|/etc/|/usr/|/var/|/proc/|/sys/)", regex=True).astype(int)

    # --- Grupo 8: HTTP request ---
    m_s = _s(method).str.upper()
    feat["method_is_get"]            = (m_s == "GET").astype(int)
    feat["method_is_post"]           = (m_s == "POST").astype(int)
    _ua = _s(ua)
    feat["ua_present"]               = (_ua.str.len() > 0).astype(int)
    feat["ua_length"]                = _ua.str.len()
    feat["ua_suspicious"]            = _ua.str.contains(_SCANNER_UA.pattern, regex=True, na=False).astype(int)
    _ct = _s(ctype)
    feat["content_type_encoded"]     = _ct.str.contains(r"(?i)application/x-www-form-urlencoded", regex=True, na=False).astype(int)
    feat["authorization_length"]     = _n(auth_len)
    feat["unusual_headers_count"]    = _n(unusual_h)
    feat["status_code"]              = _n(status)

    # --- Grupo 9: temporal (cero por defecto) ---
    for col in ["req_count_1s", "req_count_5s", "req_count_60s",
                "error_rate_4xx_60s", "endpoint_diversity_60s"]:
        feat[col] = 0.0

    return feat[FEATURE_COLS].fillna(0).astype("float32")


print(f"extract_features definida | output cols = {len(FEATURE_COLS)}")

extract_features definida | output cols = 72


In [18]:
def add_temporal_features(df_feat: pd.DataFrame, df_src: pd.DataFrame,
                            ts_col: str, ip_col: str,
                            endpoint_col: str, status_col: str) -> pd.DataFrame:
    """
    Calcula features temporales por IP usando ventanas deslizantes.
    Solo para fuentes con timestamps reales (OWASP, RussellMitchell).
    df_src debe tener ts_col (datetime), ip_col, endpoint_col, status_col.
    Los indices de df_feat y df_src deben estar alineados.
    """
    df_src = df_src.copy()
    df_src[ts_col] = pd.to_datetime(df_src[ts_col])
    df_src = df_src.sort_values(ts_col)

    req_1s  = np.zeros(len(df_src), dtype=np.float32)
    req_5s  = np.zeros(len(df_src), dtype=np.float32)
    req_60s = np.zeros(len(df_src), dtype=np.float32)
    err_60s = np.zeros(len(df_src), dtype=np.float32)
    div_60s = np.zeros(len(df_src), dtype=np.float32)

    ts_arr  = df_src[ts_col].values.astype("int64")  # nanoseconds
    ip_arr  = df_src[ip_col].values
    ep_arr  = df_src[endpoint_col].values
    st_arr  = df_src[status_col].fillna(0).astype(int).values

    NS = 1_000_000_000  # nanoseconds per second

    for i in range(len(df_src)):
        t_i  = ts_arr[i]
        ip_i = ip_arr[i]
        # Window: same IP, within [t_i - W, t_i)
        def _count_window(w_ns):
            lo = t_i - w_ns * NS
            mask = (ts_arr >= lo) & (ts_arr < t_i) & (ip_arr == ip_i)
            return int(mask.sum())

        req_1s[i]  = _count_window(1)
        req_5s[i]  = _count_window(5)
        req_60s[i] = _count_window(60)

        lo60 = t_i - 60 * NS
        m60  = (ts_arr >= lo60) & (ts_arr < t_i) & (ip_arr == ip_i)
        if m60.sum() > 0:
            err_60s[i] = float((st_arr[m60] // 100 == 4).sum()) / m60.sum()
            div_60s[i] = float(len(np.unique(ep_arr[m60])))

    # Reasignar al indice original de df_feat
    orig_idx = df_src.index
    df_feat.loc[orig_idx, "req_count_1s"]          = req_1s
    df_feat.loc[orig_idx, "req_count_5s"]          = req_5s
    df_feat.loc[orig_idx, "req_count_60s"]         = req_60s
    df_feat.loc[orig_idx, "error_rate_4xx_60s"]   = err_60s
    df_feat.loc[orig_idx, "endpoint_diversity_60s"]= div_60s
    return df_feat


def save_parquet(df: pd.DataFrame, name: str) -> Path:
    out = PROCESSED_DIR / f"{name}.parquet"
    df.to_parquet(out, index=False, engine="pyarrow")
    size_kb = out.stat().st_size / 1024
    print(f"  Guardado: {out.name}  ({len(df):,} filas, {size_kb:.0f} KB)")
    label_dist = df['label'].value_counts().sort_index().to_dict()
    inv = {v:k for k,v in LABEL.items()}
    for lbl_int, cnt in sorted(label_dist.items()):
        print(f"    label={int(lbl_int)} ({inv.get(int(lbl_int),'?'):<20}) : {cnt:>8,}")
    return out

print("Funciones auxiliares definidas")

Funciones auxiliares definidas


# Seccion 3: Payloads.csv

URLs completas etiquetadas Benign/Malicious. Malicious = XSS exclusivamente.
El payload relevante es el query string de la URL (el ataque va en el parametro GET).
Encoding: latin-1. Se eliminan 546 duplicados antes de procesar.
Timestamp: no disponible, se usa TODAY (2026-06-08).

In [19]:
df_pc = pd.read_csv(BASE_DIR / "Payloads.csv", encoding="latin-1")
df_pc = df_pc.drop_duplicates(subset="Payloads").reset_index(drop=True)

# Extraer query string de la URL
def _extract_qs(url):
    try:
        parsed = urlparse(url)
        return parsed.query if parsed.query else ""
    except Exception:
        return ""

df_pc["qs"]      = df_pc["Payloads"].apply(_extract_qs)
# Payload de analisis: usar QS si existe, si no la URL completa
df_pc["payload"] = df_pc["qs"].where(df_pc["qs"].str.len() > 0, df_pc["Payloads"])
df_pc["payload"] = df_pc["payload"].apply(lambda x: unquote(str(x)))

feats = extract_features(
    payload = df_pc["payload"],
    uri     = df_pc["Payloads"],
    qs      = df_pc["qs"],
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "payloads_csv_" + df_pc.index.astype(str))
df_out["label"]     = df_pc["Class"].map({"Benign": LABEL["legitimate"],
                                          "Malicious": LABEL["xss"]}).astype("int8")
df_out["timestamp"] = TODAY

print(f"Payloads.csv | filas={len(df_out):,} | nulos label={df_out['label'].isna().sum()}")
print(df_out[["sample_id","payload_length","encoded_char_freq","xss_marker_count","label"]].head(3).to_string())
save_parquet(df_out, "payloads_csv")

Payloads.csv | filas=42,671 | nulos label=0
        sample_id  payload_length  encoded_char_freq  xss_marker_count  label
0  payloads_csv_0            54.0                0.0               3.0      2
1  payloads_csv_1           142.0                2.0               1.0      2
2  payloads_csv_2            71.0                0.0               0.0      2
  Guardado: payloads_csv.parquet  (42,671 filas, 1329 KB)
    label=0 (legitimate          ) :   28,068
    label=2 (xss                 ) :   14,603


WindowsPath('../data/processed/payloads_csv.parquet')

# Seccion 4: payload_full.csv

Unico CSV con los cuatro vectores. Payload = fragmento de parametro (no URL completa).
Timestamp: no disponible, se usa TODAY.

In [20]:
df_pf = pd.read_csv(BASE_DIR / "payload_full.csv", encoding="utf-8")

ATTACK_MAP = {
    "norm"          : LABEL["legitimate"],
    "sqli"          : LABEL["sqli"],
    "xss"           : LABEL["xss"],
    "path-traversal": LABEL["path_traversal"],
    "cmdi"          : LABEL["command_injection"],
}

df_pf["payload_dec"] = df_pf["payload"].apply(lambda x: unquote(str(x)))

feats = extract_features(payload=df_pf["payload_dec"])

df_out = feats.copy()
df_out.insert(0, "sample_id", "payload_full_" + df_pf.index.astype(str))
df_out["label"]     = df_pf["attack_type"].map(ATTACK_MAP).astype("int8")
df_out["timestamp"] = TODAY

print(f"payload_full.csv | filas={len(df_out):,} | nulos label={df_out['label'].isna().sum()}")
print(df_out[["sample_id","payload_length","sqli_keyword_count","xss_marker_count","label"]].head(3).to_string())
save_parquet(df_out, "payload_full")

payload_full.csv | filas=31,067 | nulos label=0
        sample_id  payload_length  sqli_keyword_count  xss_marker_count  label
0  payload_full_0            14.0                 0.0               0.0      0
1  payload_full_1            12.0                 0.0               0.0      0
2  payload_full_2             5.0                 0.0               0.0      0
  Guardado: payload_full.parquet  (31,067 filas, 552 KB)
    label=0 (legitimate          ) :   19,304
    label=1 (sqli                ) :   10,852
    label=2 (xss                 ) :      532
    label=3 (path_traversal      ) :      290
    label=4 (command_injection   ) :       89


WindowsPath('../data/processed/payload_full.parquet')

# Seccion 5: command_injection.csv

Dataset binario CMDI. Los payloads estan HTML-encoded en varios casos.
Primero se decodifica HTML (html.unescape), luego URL-decode.
Timestamp: no disponible, se usa TODAY.

In [21]:
df_ci = pd.read_csv(BASE_DIR / "command injection.csv", encoding="latin-1")
df_ci = df_ci.dropna(subset=["sentence"]).drop_duplicates(subset="sentence").reset_index(drop=True)

# Doble decodificacion: HTML-unescape -> URL-decode
df_ci["payload_dec"] = df_ci["sentence"].apply(
    lambda x: unquote(html_lib.unescape(str(x)))
)

feats = extract_features(payload=df_ci["payload_dec"])

df_out = feats.copy()
df_out.insert(0, "sample_id", "cmd_injection_" + df_ci.index.astype(str))
df_out["label"]     = df_ci["Label"].map({0: LABEL["legitimate"], 1: LABEL["command_injection"]}).astype("int8")
df_out["timestamp"] = TODAY

print(f"command_injection.csv | filas={len(df_out):,} | nulos label={df_out['label'].isna().sum()}")
print(df_out[["sample_id","payload_length","shell_command_count","pipe_count","label"]].head(4).to_string())
save_parquet(df_out, "command_injection")

command_injection.csv | filas=2,059 | nulos label=0
         sample_id  payload_length  shell_command_count  pipe_count  label
0  cmd_injection_0            39.0                  1.0         0.0      4
1  cmd_injection_1            39.0                  1.0         0.0      4
2  cmd_injection_2            30.0                  2.0         0.0      4
3  cmd_injection_3            15.0                  1.0         2.0      4
  Guardado: command_injection.parquet  (2,059 filas, 73 KB)
    label=0 (legitimate          ) :    1,581
    label=4 (command_injection   ) :      478


WindowsPath('../data/processed/command_injection.parquet')

# Seccion 6: XSS_dataset.csv

Dataset binario XSS. ADVERTENCIA: Label=0 son textos de Wikipedia, no trafico HTTP.
Esto se preserva en el parquet pero se documenta en la columna sample_id con prefijo.
Timestamp: no disponible, se usa TODAY.

In [22]:
df_xss = pd.read_csv(BASE_DIR / "XSS_dataset.csv" / "XSS_dataset.csv", encoding="utf-8",
                     low_memory=False)

df_xss["payload_dec"] = df_xss["Sentence"].apply(lambda x: unquote(str(x)))

feats = extract_features(payload=df_xss["payload_dec"])

df_out = feats.copy()
df_out.insert(0, "sample_id", "xss_dataset_" + df_xss.index.astype(str))
df_out["label"]     = df_xss["Label"].map({0: LABEL["legitimate"], 1: LABEL["xss"]}).astype("int8")
df_out["timestamp"] = TODAY

print(f"XSS_dataset.csv | filas={len(df_out):,} | nulos label={df_out['label'].isna().sum()}")
print(df_out[["sample_id","payload_length","xss_marker_count","html_tag_count","label"]].head(3).to_string())
save_parquet(df_out, "xss_dataset")

XSS_dataset.csv | filas=13,686 | nulos label=0
       sample_id  payload_length  xss_marker_count  html_tag_count  label
0  xss_dataset_0           557.0               1.0             8.0      0
1  xss_dataset_1            36.0               2.0             2.0      2
2  xss_dataset_2           233.0               0.0             4.0      0
  Guardado: xss_dataset.parquet  (13,686 filas, 424 KB)
    label=0 (legitimate          ) :    6,313
    label=2 (xss                 ) :    7,373


WindowsPath('../data/processed/xss_dataset.parquet')

# Seccion 7: data_capec_multilabel.csv

Dataset con schema HTTP completo (38 columnas) y etiquetado multi-label CAPEC.
El payload de analisis se extrae de `request_http_request` (linea de peticion HTTP).
Para filas con multiples labels in-scope, se asigna prioridad: sqli > cmdi > path_traversal > xss.
Rows con `000-Normal=1` y cero labels de ataque in-scope se clasifican como legitimate.
`request_body` es null en 97% de filas: se usa como body feature cuando disponible.
Timestamp: no disponible, se usa TODAY.

In [23]:
print("Cargando data_capec_multilabel.csv (416 MB)...")
df_cap = pd.read_csv(BASE_DIR / "data_capec_multilabel.csv", low_memory=False)
df_cap = df_cap.drop_duplicates().reset_index(drop=True)
print(f"  Cargado: {len(df_cap):,} filas")

# Extraer componentes de la request line: 'GET /path?qs HTTP/1.1'
RL = df_cap["request_http_request"].fillna("")
_RL_PAT = re.compile(r'^(\S+)\s+(\S+)\s+')

def _parse_rl(s):
    m = _RL_PAT.match(s)
    if not m: return "", "", ""
    method = m.group(1)
    full   = m.group(2)
    if "?" in full:
        path, qs = full.split("?", 1)
    else:
        path, qs = full, ""
    return method, path, qs

parsed = RL.apply(_parse_rl)
df_cap["_method"] = parsed.apply(lambda x: x[0])
df_cap["_path"]   = parsed.apply(lambda x: x[1])
df_cap["_qs"]     = parsed.apply(lambda x: x[2])

# Payload = QS si existe, si no path
df_cap["_payload"] = df_cap["_qs"].where(df_cap["_qs"].str.len() > 0, df_cap["_path"])
df_cap["_payload"] = df_cap["_payload"].apply(lambda x: unquote(str(x)))

# Label: prioridad sqli > cmdi > path_traversal > xss > legitimate
IN_SCOPE = {
    "66 - SQL Injection":       LABEL["sqli"],
    "88 - OS Command Injection": LABEL["command_injection"],
    "126 - Path Traversal":     LABEL["path_traversal"],
    "242 - Code Injection":     LABEL["xss"],
}
PRIORITY = ["66 - SQL Injection", "88 - OS Command Injection",
            "126 - Path Traversal", "242 - Code Injection"]

def _assign_label(row):
    for col in PRIORITY:
        if col in row and row[col] == 1:
            return IN_SCOPE[col]
    return LABEL["legitimate"]

df_cap["_label"] = df_cap[PRIORITY].apply(_assign_label, axis=1).astype("int8")
print(f"  Distribucion labels:\n{df_cap['_label'].value_counts().sort_index().to_string()}")

Cargando data_capec_multilabel.csv (416 MB)...
  Cargado: 905,069 filas
  Distribucion labels:
_label
0    615906
1    250230
2     13838
3     18005
4      7090


In [24]:
print("Extrayendo features data_capec (puede tardar ~5 min para 907K filas)...")

feats = extract_features(
    payload = df_cap["_payload"],
    uri     = df_cap["_path"] + df_cap["_qs"].apply(lambda x: f"?{x}" if x else ""),
    qs      = df_cap["_qs"],
    method  = df_cap["_method"],
    body    = df_cap["request_body"].fillna(""),
    status  = df_cap["response_http_status_code"],
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "capec_" + df_cap.index.astype(str))
df_out["label"]     = df_cap["_label"].values
df_out["timestamp"] = TODAY

print(f"data_capec | filas={len(df_out):,}")
save_parquet(df_out, "data_capec")

del df_cap  # liberar memoria (416 MB)

Extrayendo features data_capec (puede tardar ~5 min para 907K filas)...
data_capec | filas=905,069
  Guardado: data_capec.parquet  (905,069 filas, 4853 KB)
    label=0 (legitimate          ) :  615,906
    label=1 (sqli                ) :  250,230
    label=2 (xss                 ) :   13,838
    label=3 (path_traversal      ) :   18,005
    label=4 (command_injection   ) :    7,090


# Seccion 8: modsec-learn JSON

Dos listas planas de strings:
- `legitimate_dataset.json`: 69,101 unicos (86.4% dupes en crudo)
- `malicious_dataset.json`: 30,544 SQLi, formato `p=<payload>`

Esquema: el string completo se usa como payload para el FE.
Timestamp: no disponible, se usa TODAY.

In [25]:
with open(BASE_DIR / "modsec-learn" / "legitimate_dataset.json", encoding="utf-8") as f:
    legit_raw = json.load(f)
with open(BASE_DIR / "modsec-learn" / "malicious_dataset.json", encoding="utf-8") as f:
    sqli_raw  = json.load(f)

legit_unique = list(dict.fromkeys(legit_raw))
sqli_unique  = list(dict.fromkeys(sqli_raw))

print(f"legitimate unicos: {len(legit_unique):,}")
print(f"sqli unicos      : {len(sqli_unique):,}")

# Combinar en un DataFrame unificado
df_ms = pd.DataFrame({
    "raw": legit_unique + sqli_unique,
    "label": ([LABEL["legitimate"]] * len(legit_unique) +
              [LABEL["sqli"]]        * len(sqli_unique)),
})

# Decodificar URL y extraer QS si el formato es 'p=payload'
def _decode_modsec(s):
    s = unquote(str(s))
    if s.startswith("p="):
        s = s[2:]
    return s

df_ms["payload_dec"] = df_ms["raw"].apply(_decode_modsec)
df_ms["qs"]          = df_ms["raw"].apply(
    lambda x: x if "=" in x else ""
)

feats = extract_features(
    payload = df_ms["payload_dec"],
    qs      = df_ms["qs"],
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "modsec_" + df_ms.index.astype(str))
df_out["label"]     = df_ms["label"].astype("int8")
df_out["timestamp"] = TODAY

print(f"modsec-learn | filas={len(df_out):,}")
save_parquet(df_out, "modsec_learn")

legitimate unicos: 69,101
sqli unicos      : 30,544
modsec-learn | filas=99,645
  Guardado: modsec_learn.parquet  (99,645 filas, 3059 KB)
    label=0 (legitimate          ) :   69,101
    label=1 (sqli                ) :   30,544


WindowsPath('../data/processed/modsec_learn.parquet')

# Seccion 9: PT Wordlists (Dp.txt)

Solo se procesa `Dp.txt` (1,166 payloads de Path Traversal reales para Windows).
`Deep-Travelsal.txt` contiene templates con placeholder {FILE}: se omite en este paso
por requerir expansion de archivo objetivo antes de ser valido como muestra de ataque.
Todos los samples son label=3 (path_traversal). Timestamp: TODAY.

In [26]:
PT_DIR = BASE_DIR / "omurugur Path_Travelsal_Payload_List master Payload"
dp_lines = (PT_DIR / "Dp.txt").read_text(encoding="utf-8", errors="replace").splitlines()
dp_lines = [l.strip() for l in dp_lines if l.strip()]

df_pt = pd.DataFrame({"raw": dp_lines})
df_pt["payload_dec"] = df_pt["raw"].apply(lambda x: unquote(str(x)))

feats = extract_features(
    payload = df_pt["payload_dec"],
    uri     = df_pt["payload_dec"],
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "pt_wordlist_" + df_pt.index.astype(str))
df_out["label"]     = LABEL["path_traversal"]
df_out["timestamp"] = TODAY

print(f"PT wordlists | filas={len(df_out):,}")
print(df_out[["sample_id","traversal_sequence_count","path_separator_count","sensitive_file_target","label"]].head(4).to_string())
save_parquet(df_out, "pt_wordlists")

PT wordlists | filas=1,166
       sample_id  traversal_sequence_count  path_separator_count  sensitive_file_target  label
0  pt_wordlist_0                       1.0                   3.0                    1.0      3
1  pt_wordlist_1                       2.0                   4.0                    1.0      3
2  pt_wordlist_2                       3.0                   5.0                    1.0      3
3  pt_wordlist_3                       4.0                   6.0                    1.0      3
  Guardado: pt_wordlists.parquet  (1,166 filas, 63 KB)
    label=3 (path_traversal      ) :    1,166


WindowsPath('../data/processed/pt_wordlists.parquet')

# Seccion 10: OWASP ModSecurity Logs

30 dias de logs (379.5 MB, 142,705 transacciones). Se parsean los bloques de transaccion
para extraer method, path, query_string, user-agent, status y timestamp real.
Solo se incluyen las 56,504 transacciones in-scope (sqli, xss, path_traversal, cmdi).
Las features temporales se calculan por IP usando ventanas de 1s, 5s y 60s.
Timestamp: extraido del bloque -A-- de cada transaccion.

In [27]:
OWASP_DIR  = BASE_DIR / "owasp"
log_files  = sorted(OWASP_DIR.rglob("*.anon.log"))

_TS_PAT  = re.compile(r'\[(\d{2}/\w+/\d{4}:\d{2}:\d{2}:\d{2} [+\-]\d{4})\]')
_REQ_PAT = re.compile(r'^(GET|POST|PUT|DELETE|HEAD|OPTIONS|PATCH|CONNECT|TRACE)\s+(\S+)\s+HTTP', re.M)
_UA_PAT  = re.compile(r'User-Agent:\s*(.+)', re.I)
_CT_PAT  = re.compile(r'Content-Type:\s*(.+)', re.I)
_AUTH_PAT= re.compile(r'Authorization:\s*(.+)', re.I)
_ST_PAT  = re.compile(r'HTTP/\S+\s+(\d{3})', re.M)
_IP_PAT  = re.compile(r'^(\d+\.\d+\.\d+\.\d+|\d+\.\d+\.\d+\.\[\d+\])', re.M)

def classify_owasp(tags, rule_ids):
    if "attack-sqli" in tags:                                                return "sqli"
    if "attack-xss"  in tags:                                                return "xss"
    if any(r in rule_ids for r in ["930110","930120","930130"]) \
       or "attack-lfi" in tags or "attack-rfi" in tags:                      return "path_traversal"
    if any(r in rule_ids for r in ["932150","932160"]) \
       or "attack-rce" in tags:                                              return "command_injection"
    return None

records = []

for lf in log_files:
    # Extrae fecha del directorio para fallback de timestamp
    dir_date = lf.parent.name  # e.g. '15-Aug-2025'
    try:
        fallback_ts = pd.Timestamp(dir_date)
    except Exception:
        fallback_ts = TODAY

    with open(lf, "r", encoding="utf-8", errors="replace") as fh:
        content = fh.read()

    for tx in re.split(r"(?=--[a-f0-9]+-A--)", content):
        if not tx.strip(): continue

        sec_h = re.search(r"-H--\n(.*?)(?=--[a-f0-9]+-Z--|$)", tx, re.DOTALL)
        if not sec_h: continue
        h_text = sec_h.group(1)

        tags     = set(re.findall(r'\[tag "([^"]+)"\]', h_text))
        rule_ids = set(re.findall(r'\[id "(\d+)"\]', h_text))
        label_str = classify_owasp(tags, rule_ids)
        if label_str is None: continue

        # Timestamp desde seccion A
        sec_a = re.search(r"-A--\n(.+)", tx)
        ts = fallback_ts
        if sec_a:
            tm = _TS_PAT.search(sec_a.group(1))
            if tm:
                try:
                    ts = pd.Timestamp(tm.group(1), tz="UTC").tz_localize(None)
                except Exception:
                    pass

        # Request line desde seccion B
        sec_b = re.search(r"-B--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        method, path, qs_str = "", "", ""
        body_str = ""
        ua_str, ct_str, auth_len_val = "", "", 0
        unusual = 0
        if sec_b:
            b_text = sec_b.group(1)
            rm = _REQ_PAT.search(b_text)
            if rm:
                method = rm.group(1)
                full   = rm.group(2)
                path, qs_str = (full.split("?",1) if "?" in full else (full, ""))
            ua_m  = _UA_PAT.search(b_text)
            ct_m  = _CT_PAT.search(b_text)
            auth_m= _AUTH_PAT.search(b_text)
            if ua_m:   ua_str      = ua_m.group(1).strip()
            if ct_m:   ct_str      = ct_m.group(1).strip()
            if auth_m: auth_len_val= len(auth_m.group(1).strip())
            # Cabeceras no estandar: contar lineas Header: value que no sean comunes
            std_hdrs = {"host","user-agent","accept","content-type","content-length",
                       "authorization","cookie","referer","connection","accept-encoding",
                       "accept-language","cache-control"}
            for line in b_text.splitlines():
                if ":" in line:
                    hname = line.split(":",1)[0].strip().lower()
                    if hname and hname not in std_hdrs:
                        unusual += 1

        # Body desde seccion C
        sec_c = re.search(r"-C--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        if sec_c: body_str = sec_c.group(1)[:2000]

        # Status desde seccion F
        sec_f = re.search(r"-F--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        status_val = 0
        if sec_f:
            sm = _ST_PAT.search(sec_f.group(1))
            if sm: status_val = int(sm.group(1))

        # IP desde seccion A
        ip_str = "0.0.0.0"
        if sec_a:
            ipm = _IP_PAT.search(sec_a.group(1))
            if ipm: ip_str = ipm.group(1)

        payload_dec = unquote(qs_str) if qs_str else unquote(path)

        records.append({
            "label_str": label_str, "timestamp": ts,
            "method": method, "path": path, "qs": qs_str,
            "payload": payload_dec, "body": body_str,
            "ua": ua_str, "ctype": ct_str, "auth_len": auth_len_val,
            "unusual_h": unusual, "status": status_val, "ip": ip_str,
        })

df_ow = pd.DataFrame(records)
print(f"OWASP in-scope transacciones: {len(df_ow):,}")
print(df_ow["label_str"].value_counts().to_string())

OWASP in-scope transacciones: 56,504
label_str
path_traversal       49849
sqli                  4064
command_injection     2331
xss                    260


In [28]:
feats = extract_features(
    payload  = df_ow["payload"],
    uri      = (df_ow["path"] + df_ow["qs"].apply(lambda x: f"?{x}" if x else "")),
    qs       = df_ow["qs"],
    method   = df_ow["method"],
    body     = df_ow["body"],
    ua       = df_ow["ua"],
    ctype    = df_ow["ctype"],
    auth_len = df_ow["auth_len"],
    unusual_h= df_ow["unusual_h"],
    status   = df_ow["status"],
)

# Features temporales por IP (ventanas deslizantes)
print("Calculando features temporales OWASP...")
feats = add_temporal_features(
    feats, df_ow,
    ts_col="timestamp", ip_col="ip",
    endpoint_col="path", status_col="status",
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "owasp_" + df_ow.index.astype(str))
df_out["label"]     = df_ow["label_str"].map(LABEL).astype("int8")
df_out["timestamp"] = df_ow["timestamp"].values

print(f"OWASP | filas={len(df_out):,}")
print(df_out[["sample_id","traversal_sequence_count","sqli_keyword_count",
              "req_count_60s","label"]].head(3).to_string())
save_parquet(df_out, "owasp_logs")

del df_ow, records

Calculando features temporales OWASP...
OWASP | filas=56,504
  sample_id  traversal_sequence_count  sqli_keyword_count  req_count_60s  label
0   owasp_0                       0.0                 0.0            0.0      3
1   owasp_1                       0.0                 0.0            0.0      3
2   owasp_2                       0.0                 0.0            0.0      3
  Guardado: owasp_logs.parquet  (56,504 filas, 823 KB)
    label=1 (sqli                ) :    4,064
    label=2 (xss                 ) :      260
    label=3 (path_traversal      ) :   49,849
    label=4 (command_injection   ) :    2,331


# Seccion 11: RussellMitchell (legitimate only)

Access logs Apache CLF del escenario WordPress. Se usa solo como fuente de trafico legitimo:
- `.log.1`, `.log.3`, `.log.4`: trafico sin etiquetar (todos asumidos legitimate)
- `.log.2`: solo las 821 lineas sin label (legitimate confirmado)
Timestamp: extraido del formato CLF `[23/Jan/2022:10:15:30 +0000]`.

In [29]:
RM_LOG_DIR    = BASE_DIR / "russellmitchell" / "gather" / "intranet_server" / "logs" / "apache2"
RM_LABELS_DIR = BASE_DIR / "russellmitchell" / "labels" / "intranet_server" / "logs" / "apache2"

_CLF_PAT  = re.compile(
    r'(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) \S+" (\d+) \S+'
    r'(?: "[^"]*" "([^"]*)")?' # referer + UA opcionalmente
)
_MONTH = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
          "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}

def _parse_clf_ts(s):
    # '23/Jan/2022:10:15:30 +0000'
    try:
        d, rest = s.split(":" ,1)
        day, mon, yr = d.split("/")
        h, mi, sec_tz = rest.split(":",2)
        sec = sec_tz.split(" ")[0]
        return pd.Timestamp(int(yr), _MONTH[mon], int(day), int(h), int(mi), int(sec))
    except Exception:
        return TODAY

# Cargar lineas etiquetadas de log.2 para filtrar las labeled (recon/attack)
label_file  = RM_LABELS_DIR / "intranet.smith.russellmitchell.com-access.log.2"
labeled_lines = set()
with open(label_file) as lf:
    for line in lf:
        obj = json.loads(line)
        labeled_lines.add(obj["line"])

records_rm = []

for fname in [
    "intranet.smith.russellmitchell.com-access.log.2",
    "intranet.smith.russellmitchell.com-access.log.1",
    "intranet.smith.russellmitchell.com-access.log.3",
    "intranet.smith.russellmitchell.com-access.log.4",
]:
    fpath = RM_LOG_DIR / fname
    if not fpath.exists(): continue
    with open(fpath, encoding="utf-8", errors="replace") as fh:
        for lineno, line in enumerate(fh, start=1):
            # Para log.2: saltar lineas con labels de ataque
            if fname.endswith(".log.2") and lineno in labeled_lines:
                continue
            m = _CLF_PAT.match(line.strip())
            if not m: continue
            ip, ts_str, method, full_path, status = (
                m.group(1), m.group(2), m.group(3), m.group(4), int(m.group(5))
            )
            ua_str = m.group(6) or ""
            path, qs_str = (full_path.split("?",1) if "?" in full_path else (full_path, ""))
            records_rm.append({
                "ip": ip, "timestamp": _parse_clf_ts(ts_str),
                "method": method, "path": path, "qs": qs_str,
                "payload": unquote(qs_str) if qs_str else unquote(path),
                "ua": ua_str, "status": status,
            })

df_rm = pd.DataFrame(records_rm)
print(f"RussellMitchell legitimate | filas={len(df_rm):,}")

feats = extract_features(
    payload = df_rm["payload"],
    uri     = df_rm["path"] + df_rm["qs"].apply(lambda x: f"?{x}" if x else ""),
    qs      = df_rm["qs"],
    method  = df_rm["method"],
    ua      = df_rm["ua"],
    status  = df_rm["status"],
)

feats = add_temporal_features(
    feats, df_rm,
    ts_col="timestamp", ip_col="ip",
    endpoint_col="path", status_col="status",
)

df_out = feats.copy()
df_out.insert(0, "sample_id", "russellmitchell_" + df_rm.index.astype(str))
df_out["label"]     = LABEL["legitimate"]
df_out["timestamp"] = df_rm["timestamp"].values

print(f"RussellMitchell | filas={len(df_out):,}")
save_parquet(df_out, "russellmitchell")

RussellMitchell legitimate | filas=3,435
RussellMitchell | filas=3,435
  Guardado: russellmitchell.parquet  (3,435 filas, 99 KB)
    label=0 (legitimate          ) :    3,435


WindowsPath('../data/processed/russellmitchell.parquet')

# Seccion 12: Resumen de parquets generados

Verificacion de que todos los archivos existen, tienen el schema correcto y
no hay labels inesperados ni columnas faltantes.

In [30]:
import pyarrow.parquet as pq

expected_files = [
    "payloads_csv", "payload_full", "command_injection",
    "xss_dataset", "data_capec", "modsec_learn",
    "pt_wordlists", "owasp_logs", "russellmitchell",
]

inv_label = {v: k for k, v in LABEL.items()}
total_rows = 0

print(f"{'Archivo':<25} {'Filas':>8}  {'MB':>6}  {'Cols':>5}  Label distribution")
print("-" * 100)

for name in expected_files:
    fpath = PROCESSED_DIR / f"{name}.parquet"
    if not fpath.exists():
        print(f"  FALTA: {name}.parquet")
        continue
    df_tmp = pd.read_parquet(fpath)
    size_mb = fpath.stat().st_size / (1024**2)
    n_rows  = len(df_tmp)
    n_cols  = len(df_tmp.columns)
    total_rows += n_rows
    missing_feats = [c for c in FEATURE_COLS if c not in df_tmp.columns]
    dist = df_tmp["label"].value_counts().sort_index()
    dist_str = "  ".join(f"{inv_label.get(int(k),k)}:{v:,}" for k, v in dist.items())
    warn = f" [FALTAN COLS: {missing_feats}]" if missing_feats else ""
    print(f"  {name:<23} {n_rows:>8,}  {size_mb:>6.1f}  {n_cols:>5}  {dist_str}{warn}")

print()
print(f"Total filas en processed/: {total_rows:,}")
print(f"Schema esperado: sample_id + {len(FEATURE_COLS)} features + label + timestamp = {3 + len(FEATURE_COLS)} columnas")

Archivo                      Filas      MB   Cols  Label distribution
----------------------------------------------------------------------------------------------------
  payloads_csv              42,671     1.3     75  legitimate:28,068  xss:14,603
  payload_full              31,067     0.5     75  legitimate:19,304  sqli:10,852  xss:532  path_traversal:290  command_injection:89
  command_injection          2,059     0.1     75  legitimate:1,581  command_injection:478
  xss_dataset               13,686     0.4     75  legitimate:6,313  xss:7,373
  data_capec               905,069     4.7     75  legitimate:615,906  sqli:250,230  xss:13,838  path_traversal:18,005  command_injection:7,090
  modsec_learn              99,645     3.0     75  legitimate:69,101  sqli:30,544
  pt_wordlists               1,166     0.1     75  path_traversal:1,166
  owasp_logs                56,504     0.8     75  sqli:4,064  xss:260  path_traversal:49,849  command_injection:2,331
  russellmitchell           